# Pobieranie i scalanie danych pomiarowych ze stacji naziemnych IMGW (Balice, Obserwatorium)

## Konfiguracja i importowanie bibliotek

In [ ]:
import os
import sys
import zipfile
from pathlib import Path
import pandas as pd
import requests

# Dodanie katalogu głównego projektu do sys.path
root_dir = Path("..").resolve()
if str(root_dir) not in sys.path:
  sys.path.append(str(root_dir))

# Import ścieżki surowych danych ze wspólnego modułu konfiguracyjnego
from src.config import IMGW_DATA_DIR, BALICE_PATH, OBSERWATORIUM_PATH

# Przygotowanie katalogów roboczych na dane IMGW
BALICE_DIR = IMGW_DATA_DIR / "balice"
OBSERWATORIUM_DIR = IMGW_DATA_DIR / "obserwatorium"

BALICE_DIR.mkdir(parents=True, exist_ok=True)
OBSERWATORIUM_DIR.mkdir(parents=True, exist_ok=True)
print("Katalogi dla danych IMGW zostały przygotowane.")

Katalogi dla danych IMGW zostały przygotowane.


## Pobieranie i scalanie danych


### Kraków - Balice (stacja synoptyczna)

In [ ]:
station_id = "566"
station_name = "KRAKOW_BALICE"
base_url = "https://danepubliczne.imgw.pl/data/dane_pomiarowo_obserwacyjne/dane_meteorologiczne/terminowe/synop/"
start_year = 2000
end_year = 2025

print("--- 1. POBIERANIE DANYCH DLA STACJI KRAKÓW-BALICE ---")
for year in range(start_year, end_year + 1):
    year_file = BALICE_DIR / f"{station_name}_{year}.csv"
    if year_file.exists():
        print(f"Pominięto: {year_file.name}. Plik już istnieje.")
        continue

    year_folder = "1996_2000" if 1996 <= year <= 2000 else str(year)
    zip_filename = f"1996_2000_{station_id}_s.zip" if year == 2000 else f"{year}_{station_id}_s.zip"
    url = f"{base_url}{year_folder}/{zip_filename}"

    try:
        r = requests.get(url, timeout=25)
        if r.status_code == 200:
            temp_zip = BALICE_DIR / f"temp_{zip_filename}"
            with open(temp_zip, "wb") as f:
                f.write(r.content)

            with zipfile.ZipFile(temp_zip, "r") as z:
                csvs = [f for f in z.namelist() if station_id in f and f.endswith(".csv")]
                if csvs:
                with z.open(csvs[0]) as f_csv:
                    df = pd.read_csv(f_csv, header=None, encoding="iso-8859-2", low_memory=False)
                    # Kolumny IMGW Synop: 2=rok, 3=miesiąc, 4=dzień, 5=godzina, 29=temperatura
                    df_sub = df[[2, 3, 4, 5, 29]].copy()
                    df_sub.columns = ["year", "month", "day", "hour", "temp"]
                    df_sub["temp"] = pd.to_numeric(df_sub["temp"], errors="coerce")
                    df_year = df_sub[df_sub["year"] == year].copy()

                    df_year.to_csv(year_file, index=False)
                    print(f"Balice {year}: zapisano {year_file.name} ({len(df_year)} wierszy)")
            temp_zip.unlink(missing_ok=True)
        else:
            print(f"Balice {year}: brak na serwerze (kod {r.status_code})")
    except Exception as e:
        print(f"Balice {year}: błąd ({e})")

print("Zakończono pobieranie danych dla Balic.")

# Scalanie plików rocznych Balic do jednego CSV
yearly_files = sorted(BALICE_DIR.glob(f"{station_name}_*.csv"))
if yearly_files:
    df_all = pd.concat([pd.read_csv(f) for f in yearly_files], ignore_index=True)
    df_all.sort_values(by=['year', 'month', 'day', 'hour'], inplace=True)
    df_all.to_csv(BALICE_PATH, index=False)
    print(f"Utworzono plik zbiorczy Balic: {BALICE_PATH} ({len(df_all):,} wierszy)\n")

### Kraków - Obserwatorium (stacja klimatyczna)

In [ ]:
station_id = 250190390
station_name = "KRAKOW_OBSERWATORIUM"
base_url = "https://danepubliczne.imgw.pl/data/dane_pomiarowo_obserwacyjne/dane_meteorologiczne/terminowe/klimat/"
start_year = 2000
end_year = 2025

print("--- 2. POBIERANIE DANYCH DLA STACJI KRAKÓW-OBSERWATORIUM ---")
for year in range(start_year, end_year + 1):
    year_file = OBSERWATORIUM_DIR / f"{station_name}_{year}.csv"
    if year_file.exists():
        continue

    year_records = []
    folders = [str(year), "1996_2000"] if year == 2000 else [str(year)]

    # 1. Próba pobrania paczek miesięcznych (_k.zip)
    found_monthly = False
    for month in range(1, 13):
        for folder in folders:
            zip_filename = f"{year}_{month:02d}_k.zip"
            url = f"{base_url}{folder}/{zip_filename}"
            try:
                r = requests.get(url, timeout=15)
                if r.status_code == 200:
                    temp_zip = download_dir / f"temp_{zip_filename}"
                    with open(temp_zip, "wb") as f:
                        f.write(r.content)
                    with zipfile.ZipFile(temp_zip, "r") as z:
                        csvs = [f for f in z.namelist() if f.endswith(".csv")]
                        if csvs:
                            with z.open(csvs[0]) as f_csv:
                                df = pd.read_csv(f_csv, header=None, encoding='iso-8859-2', low_memory=False)
                                df_st = df[df[0] == station_id][[2, 3, 4, 5, 6]].copy()
                                df_st.columns = ['year', 'month', 'day', 'hour', 'temp']
                                df_st['temp'] = pd.to_numeric(df_st['temp'], errors='coerce')
                                df_st = df_st[df_st['year'] == year]
                                if not df_st.empty:
                                    year_records.append(df_st)
                                    found_monthly = True
                    temp_zip.unlink(missing_ok=True)
                    break
            except Exception:
                pass

    # 2. Próba paczki rocznej (jeśli w danym roku nie było paczek miesięcznych)
    if not found_monthly:
        for folder in folders:
            zip_filename = f"{year}_k.zip"
            url = f"{base_url}{folder}/{zip_filename}"
            try:
                r = requests.get(url, timeout=20)
                if r.status_code == 200:
                    temp_zip = OBSERWATORIUM_DIR / f"temp_{zip_filename}"
                    with open(temp_zip, "wb") as f:
                        f.write(r.content)
                    with zipfile.ZipFile(temp_zip, "r") as z:
                        csvs = [f for f in z.namelist() if f.endswith(".csv")]
                        if csvs:
                            with z.open(csvs[0]) as f_csv:
                                df = pd.read_csv(f_csv, header=None, encoding='iso-8859-2', low_memory=False)
                                df_st = df[df[0] == station_id][[2, 3, 4, 5, 6]].copy()
                                df_st.columns = ['year', 'month', 'day', 'hour', 'temp']
                                df_st['temp'] = pd.to_numeric(df_st['temp'], errors='coerce')
                                df_st = df_st[df_st['year'] == year]
                                if not df_st.empty:
                                    year_records.append(df_st)
                    temp_zip.unlink(missing_ok=True)
                    break
            except Exception:
                pass

    # Zapis pliku rocznego dla Obserwatorium
    if year_records:
        df_year = pd.concat(year_records, ignore_index=True)
        df_year.sort_values(by=['month', 'day', 'hour'], inplace=True)
        df_year.to_csv(year_file, index=False)
        print(f"Obserwatorium {year}: zapisano {year_file.name} ({len(df_year)} wierszy)")
    else:
        print(f"Obserwatorium {year}: nie odnaleziono danych")

# Scalanie plików rocznych Obserwatorium do jednego CSV
yearly_files = sorted(OBSERWATORIUM_DIR.glob(f"{station_name}_*.csv"))
if yearly_files:
    df_all = pd.concat([pd.read_csv(f) for f in yearly_files], ignore_index=True)
    df_all.sort_values(by=['year', 'month', 'day', 'hour'], inplace=True)
    df_all.to_csv(OBSERWATORIUM_PATH, index=False)
    print(f"Utworzono plik zbiorczy Obserwatorium: {OBSERWATORIUM_PATH} ({len(df_all):,} wierszy)\n")


## Wczytanie scalonych zbiorów i weryfikacja danych

In [2]:
# 1. Weryfikacja Balic
if BALICE_PATH.exists():
  df_balice = pd.read_csv(BALICE_PATH)
  print("=== STACJA KRAKÓW-BALICE ===")
  print(f"Liczba pomiarów: {len(df_balice):,}")
  print(f"Zakres lat: {df_balice['year'].min()} - {df_balice['year'].max()}")
  print(df_balice.head(), "\n")

# 2. Weryfikacja Obserwatorium
if OBSERWATORIUM_PATH.exists():
  df_obs = pd.read_csv(OBSERWATORIUM_PATH)
  print("=== STACJA KRAKÓW-OBSERWATORIUM ===")
  print(f"Liczba pomiarów: {len(df_obs):,}")
  print(f"Zakres lat: {df_obs['year'].min()} - {df_obs['year'].max()}")
  print(df_obs.head())

=== STACJA KRAKÓW-BALICE ===
Liczba pomiarów: 227,928
Zakres lat: 2000 - 2025
   year  month  day  hour  temp
0  2000      1    1     0 -11.4
1  2000      1    1     1 -12.0
2  2000      1    1     2 -12.5
3  2000      1    1     3 -12.0
4  2000      1    1     4 -11.0 

=== STACJA KRAKÓW-OBSERWATORIUM ===
Liczba pomiarów: 28,491
Zakres lat: 2000 - 2025
   year  month  day  hour  temp
0  2000      1    1     6  -7.4
1  2000      1    1    12  -2.3
2  2000      1    1    18  -6.7
3  2000      1    2     6  -8.9
4  2000      1    2    12  -5.2
